# Занятие 6, пара 2. Поиск и классификация изображений по содержанию


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Найти похожее — а похожее в каком смысле?

Каталог можно искать по цвету, форме, фактуре или смыслу. Признак определяет, что алгоритм
замечает и что теряет. Начнём с поиска фрагмента настоящей фотографии, затем построим пример,
на котором цветовая гистограмма принципиально не различает две картинки.

Во второй части вернёмся к знакомому из ML ближайшему соседу: сравним два признака на validation,
зафиксируем выбор и один раз посмотрим test. Никаких нейросетей и GPU здесь не требуется.
Нужны массивы, гистограмма как счёт частот, расстояние между векторами; новые операции объясняются по пути.
Результат — выдача top-k с изображениями, проверенный контрпример и небольшой честный классификационный опыт.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


### 1. Галерея, которую действительно можно рассмотреть

Возьмём 12 неперекрывающихся фрагментов фотографии чашки: сетка 3×4. Нижние лишние строки
обрежем явно, размеры каждого фрагмента одинаковы. Здесь ID — номер фрагмента, не класс предмета.
На query сделаем изменение яркости одного фрагмента, чтобы искать похожую картинку, а не сравнивать имена файлов.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

photo = np.load(DATA_DIR / 'segmentation' / 'coffee.npy',allow_pickle=False).astype(float)/255
gallery = []
for row in range(3):
    for col in range(4):
        gallery.append(photo[row*132:(row+1)*132,col*150:(col+1)*150].copy())
gallery = np.array(gallery)
query = np.clip(gallery[6]*0.9,0,1)
fig,axes = plt.subplots(3,4,figsize=(12,10))
for i,axis in enumerate(axes.ravel()):
    axis.imshow(gallery[i])
    axis.set_title('Фрагмент '+str(i))
    axis.axis('off')
plt.show()
plt.figure(figsize=(5,4))
plt.imshow(query)
plt.title('Запрос')
plt.axis('off')
plt.show()


### 2. Посчитаем, сколько каких цветов

Гистограмма разбивает диапазон на интервалы и считает, сколько значений попало в каждый.
`np.histogram(values,bins=4,range=(0,1))` возвращает счётчики и границы интервалов.
Для RGB построим три отдельных гистограммы. Это упрощённый признак: он теряет и расположение
пикселей, и совместные сочетания каналов. Но для начала его легко понять и проверить.


In [ ]:
values = np.array([0.1,0.2,0.3,0.8,0.9])
counts,edges = np.histogram(values,bins=4,range=(0,1))
print('Границы:',edges)
print('Количество:',counts)
print('Доли:',counts/counts.sum())


Нормируем счётчики на число пикселей, чтобы размер изображения сам по себе не менял
признак. `np.concatenate` ставит три одномерных массива подряд. Получим 24 числа при 8 интервалах на канал.
Каждый канал суммируется в 1; весь вектор — в 3. Это выбранная нормировка, не вероятность класса.

Сначала соберём признак каждой картинки обычным циклом. `np.linalg.norm(..., axis=1)` посчитает евклидово расстояние по каждой строке: корень из суммы квадратов разностей координат. `np.argsort` вернёт индексы от меньшего расстояния к большему; `kind="stable"` сохраняет исходный порядок при равенстве. Для выдачи возьмём первые три индекса, а не три самых маленьких числа расстояния.


In [ ]:
def color_histogram(picture,bins=8):
    parts = []
    for channel in range(3):
        counts,_ = np.histogram(picture[:,:,channel],bins=bins,range=(0,1))
        parts.append(counts/counts.sum())
    return np.concatenate(parts)

query_feature = color_histogram(query)
feature_rows = []
for picture in gallery:
    feature_rows.append(color_histogram(picture))
features = np.array(feature_rows)
distances = np.linalg.norm(features-query_feature,axis=1)
ranking = np.argsort(distances,kind='stable')
print('Первые три:',ranking[:3], 'расстояния:',distances[ranking[:3]])


Расстояние измеряет различие выбранных признаков. Оно не является вероятностью правильного ответа. Посмотрим на сами найденные картинки: по одному списку номеров объяснить качество поиска трудно.


In [ ]:
fig,axes = plt.subplots(1,4,figsize=(14,4))
axes[0].imshow(query)
axes[0].set_title('Запрос')
for axis,index in zip(axes[1:
    ],ranking[:3]):
    axis.imshow(gallery[index])
    axis.set_title(f'ID{index}; d={distances[index]:.3f}')
for axis in axes:
    axis.axis('off')
plt.show()


В этой задаче есть известный исходный фрагмент 6. Его место полезно проверить, но это
не тест обобщения на новые фотографии: запрос получен из той же фотографии и почти того же фрагмента.
Найдите глазами объяснение двум остальным соседям. Что совпало: предмет или только цвет?

### 3. Добавим немного информации о расположении

Разделим картинку на четыре прямоугольника 2×2 и посчитаем прежний признак в каждом.
Склеим в фиксированном порядке: верхний левый, верхний правый, нижний левый, нижний правый.
Теперь верх и низ различимы, зато сильный сдвиг объекта может испортить сходство.
Мы выбираем компромисс под задачу, а не универсальный лучший признак.


In [ ]:
def spatial_histogram(picture,bins=8):
    height,width = picture.shape[:2]
    parts = []
    for row in range(2):
        for col in range(2):
            patch = picture[row*height//2:(row+1)*height//2,col*width//2:(col+1)*width//2]
            parts.append(color_histogram(patch,bins))
    return np.concatenate(parts)


### ✏️ Ваш ход — доказать потерю информации

Создайте две **разные** картинки 128×192, в каждой половина
пикселей красная, половина синяя. Расположите цвета так, чтобы общие гистограммы совпали,
а пространственные признаки различались. Сохраните `my_a`, `my_b` и покажите рядом.
Проверьте оба расстояния. В выводе объясните, почему увеличение числа интервалов общей
гистограммы не вернёт потерянное расположение. Не используйте случайный шум: нужен понятный контрпример.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('06.2.1', my_a, my_b, color_histogram, spatial_histogram)


**Мой вывод:** …


### 4. Три картинки или три разных объекта?

В каталоге у одного товара бывает несколько кадров. Обычный top3 может целиком занять один товар,
хотя пользователь хотел три варианта. После ранжирования можно пройти список по порядку
и оставить первое появление каждого object_id. Сам признак менять ради этого не требуется.
Ниже — входные данные для задачи; алгоритм выбора предстоит написать вам.


### ✏️ Ваш ход — собрать разнообразную выдачу

Напишите `my_unique_top(ranking,object_ids,k)`: возвращает индексы
первых k **разных объектов** в уже отсортированном ranking. Порядок важен; берём лучший кадр каждого.
Проверьте ranking=[0,1,2,3,4], object_ids=['A','A','B','C','B'],k=3 и случай, когда разных объектов меньше k.
Не берите сначала ranking[:k]: тогда можете потерять подходящие объекты дальше по списку.
В выводе объясните, почему одинаковая картинка и одинаковый object_id — не одно и то же.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('06.2.2', my_unique_top)


**Мой вывод:** …


### 5. Из поиска — в классификацию

Если у соседей известны классы, ближайший сосед может предложить свой класс новому изображению.
Это 1-NN: обучение здесь состоит в сохранении размеченных признаков, а не подборе весов нейросети.
Плохой признак даст плохой ответ даже при безошибочном коде.

Новый набор — **синтетические обложки**, а не фрагменты чашки. Класс 0:верх/низ, класс 1:лево/право.
Каждый object_id имеет два варианта яркости. Все варианты одного объекта должны остаться в одной части.
Train — база соседей; validation — выбор признака; test — последняя проверка после выбора.

`np.flatnonzero(mask)` вернёт индексы True в одномерной маске. Так выберем несколько учебных изображений для показа. У каждого изображения есть отдельно класс (что распознаём), object_id (какой исходный объект снят) и split (где разрешено использовать этот снимок). Эти три поля отвечают на разные вопросы.


In [ ]:
dataset = np.load(DATA_DIR / 'segmentation' / 'covers.npz',allow_pickle=False)
images = dataset['images']
targets = dataset['labels']
object_ids = dataset['object_ids']
split = dataset['split']
train_mask = split=='train'
val_mask = split=='validation'
test_mask = split=='test'
fig,axes = plt.subplots(2,4,figsize=(12,5))
for axis,index in zip(axes.ravel(),np.flatnonzero(train_mask)[:
    8]):
    axis.imshow(images[index])
    axis.set_title(f'{object_ids[index]}, класс{targets[index]}')
    axis.axis('off')
plt.show()


Сначала посмотрим, как ответить для **одного** вектора: расстояния до train → индекс минимума → метка.
Ни validation, ни test не добавляем в базу соседей.


In [ ]:
train_images = images[train_mask]
train_targets = targets[train_mask]
train_rows = []
for picture in train_images:
    train_rows.append(color_histogram(picture))
train_features = np.array(train_rows)
one_val = color_histogram(images[val_mask][0])
distances = np.linalg.norm(train_features-one_val,axis=1)
nearest = np.argmin(distances)
print('Предсказанный класс:',train_targets[nearest])


Теперь повторим это для набора запросов. Никакого нового метода внутри функции нет.
`feature_function` передаёт выбранный способ описания картинки. Доля верных ответов — среднее
булевых сравнений. Она зависит от нашего набора и сама не обещает качество на настоящих обложках.


In [ ]:
def predict_nearest(train_images,train_labels,query_images,feature_function):
    train_rows = []
    for picture in train_images:
        train_rows.append(feature_function(picture))
    train_vectors = np.array(train_rows)
    predictions = []
    for picture in query_images:
        vector = feature_function(picture)
        distances = np.linalg.norm(train_vectors-vector,axis=1)
        nearest = np.argmin(distances)
        predictions.append(train_labels[nearest])
    return np.asarray(predictions)

val_global = predict_nearest(train_images,train_targets,images[val_mask],color_histogram)
val_spatial = predict_nearest(train_images,train_targets,images[val_mask],spatial_histogram)
print('Validation, общий цвет:',np.mean(val_global==targets[val_mask]))
print('Validation, цвет по частям:',np.mean(val_spatial==targets[val_mask]))


### ✏️ Ваш ход — раскрыть утечку, пока оценка не стала красивой

Коллега предлагает: первые изображения каждого
объекта взять в train, вторые — в test. Напишите `my_overlap(train_ids,test_ids)`, которая возвращает
множество общих object_id. Проверьте предложение на ['a','b','c'] и ['a','b','c'], затем реальное
разбиение предоставленного набора. Почему разные файлы всё ещё могут быть зависимыми примерами?
Опишите, что именно проверял бы тест со вторым вариантом яркости знакомого объекта.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('06.2.3', my_overlap)


**Мой вывод:** …


### 6. Зафиксируем выбор и только теперь откроем test

Для задачи расположения цветовых половин выбираем пространственный признак по validation.
Фиксируем число интервалов 8 и правило 1-NN. Ниже независимая основная ячейка, ответ упражнения не нужен.
После просмотра test не меняем признак ради красивого числа: следующая идея требует новой
процедуры проверки, а не повторного подбора по тому же test.


In [ ]:
assert not (set(object_ids[train_mask]) & set(object_ids[test_mask]))
assert not (set(object_ids[train_mask]) & set(object_ids[val_mask]))
assert not (set(object_ids[val_mask]) & set(object_ids[test_mask]))
chosen_feature = spatial_histogram
test_prediction = predict_nearest(train_images,train_targets,images[test_mask],chosen_feature)
test_accuracy = np.mean(test_prediction==targets[test_mask])
print('Test после выбора:',test_accuracy)
fig,axes = plt.subplots(2,4,figsize=(12,5))
for axis,picture,predicted,truth in zip(axes.ravel(),images[test_mask],test_prediction,targets[test_mask]):
    axis.imshow(picture)
    axis.set_title(f'Предсказано{predicted}; истина{truth}')
    axis.axis('off')
plt.show()


Даже идеальный результат означает лишь, что в этом контролируемом наборе признак передаёт
нужное расположение. Реальный поиск по смыслу требует более сложных данных и признаков;
эмбеддинги нейросетей обсудим позже. Они тоже не освобождают от вопроса о данных и проверке ошибок.

**Резерв 10–15 минут:** сделайте запросом однотонную серую картинку. Ближайший сосед всё равно
назначит ей класс — такого исхода требует правило. Предложите условие отказа от ответа по расстоянию.
Обсудите, на каких validation-примерах выбирать порог; не подбирайте его по test.

**Итог:** сохраните галерею выдачи, контрпример для гистограммы и вывод по классификации.
Обязательно разделите три утверждения: нашли похожий фрагмент; различили два класса генератора;
научились работать с реальными новыми объектами. Последнее сегодня ещё не проверено.


**Почитать по желанию:** Richard Szeliski,
[Computer Vision: Algorithms and Applications](https://szeliski.org/Book/).
Начните с иллюстраций к интересующему методу и попробуйте связать одну из них с нашим опытом.
Не нужно сразу читать всё или разбирать все формулы; ссылка ведёт на сайт автора.
